# CrashVLM: GPU runs (Colab)

Runs the steps that need a GPU. Every step writes its results to Google Drive and **can be re-run
after a disconnect**: finished clips are skipped.

**Before you start**
1. Runtime → Change runtime type → **GPU** (L4 recommended; T4 works for steps A–C).
2. Upload the folder `data` from `~/crashvlm_colab_upload/` on your Mac to **MyDrive/crashvlm/**,
   so you have `MyDrive/crashvlm/data/annotation/pass_a/images.zip` etc.
3. Add your GitHub token as a Colab secret named **GITHUB_TOKEN** (key icon in the left sidebar),
   and enable notebook access.

| Step | What | Time (L4) |
|---|---|---|
| A | SigLIP 2 so400m embeddings | ~5 min |
| B | Qwen3.5-4B zero-shot reports | ~15 min |
| C | Qwen3-VL-8B (4-bit) zero-shot reports | ~20 min |
| D | LoRA fine-tuning + evaluation | ~1-2 h |
| E | Learning curve (optional) | ~2 h |


In [ ]:
# 0. GPU check, Drive, code
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["CRASHVLM_DATA"] = "/content/drive/MyDrive/crashvlm/data"
assert os.path.exists(os.environ["CRASHVLM_DATA"] + "/annotation/pass_a/images.zip"), (
    "Data not found: upload the 'data' folder to MyDrive/crashvlm/")

token = userdata.get("GITHUB_TOKEN")  # key icon -> secret 'GITHUB_TOKEN', notebook access ON
repo = "/content/crashvlm"
if not os.path.exists(repo + "/scripts"):
    r = subprocess.run(["git", "clone", f"https://{token}@github.com/Faheem8585/crashvlm.git", repo],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed (check the token): " + r.stderr.replace(token, "***"))
else:
    subprocess.run(["git", "-C", repo, "pull", "-q"], check=True)
os.chdir(repo)
print("code ready:", os.getcwd())
!pip -q install -e ".[ml]" bitsandbytes
!pip -q install -U transformers


## A. SigLIP 2 so400m frame embeddings

In [ ]:
!python scripts/extract_embeddings.py

## B. Qwen3.5-4B zero-shot crash reports (46 test clips)

In [ ]:
!python scripts/run_vlm.py vlm.model=Qwen/Qwen3.5-4B vlm.run_name=qwen3.5-4b_zs

In [ ]:
# quick look at one report
import json
line = open(os.environ["CRASHVLM_DATA"] + "/vlm/qwen3.5-4b_zs.jsonl").readline()
r = json.loads(line); print(r["parse_ok"], r["issues"]); print(r["text"][:800])

## C. Qwen3-VL-8B in 4-bit

In [ ]:
!python scripts/run_vlm.py vlm.model=Qwen/Qwen3-VL-8B-Instruct vlm.load_in_4bit=true vlm.run_name=qwen3-vl-8b-4bit_zs

## D. LoRA fine-tuning of Qwen3.5-4B, then evaluation on the same test clips

In [ ]:
!python scripts/train_lora.py vlm.model=Qwen/Qwen3.5-4B vlm.run_name=qwen3.5-4b_lora

In [ ]:
!python scripts/run_vlm.py vlm.model=Qwen/Qwen3.5-4B vlm.adapter=$CRASHVLM_DATA/lora/qwen3.5-4b_lora/adapter vlm.run_name=qwen3.5-4b_lora

## E. (optional) Learning curve: 25 % and 50 % of the training tests

In [ ]:
for f in ["0.25", "0.5"]:
    !python scripts/train_lora.py vlm.model=Qwen/Qwen3.5-4B sft.train_file=train_f{f}.jsonl vlm.run_name=qwen3.5-4b_lora_f{f}
    !python scripts/run_vlm.py vlm.model=Qwen/Qwen3.5-4B vlm.adapter=$CRASHVLM_DATA/lora/qwen3.5-4b_lora_f{f}/adapter vlm.run_name=qwen3.5-4b_lora_f{f}

## Done
Everything is in **MyDrive/crashvlm/data**: `features/`, `vlm/`, `lora/`. Download `features/`,
`vlm/` and `lora/*/train_log.json` into `~/crashvlm_data/` on the Mac, then tell Claude:
*"GPU results are in"*.
